## 1. Data loading and first checks

The dataset contains 541,909 transaction lines from a UK-based online gift retailer (1 Dec 2010 – 9 Dec 2011).

- **Duplicates:** 5,268 fully identical rows (~1%) were removed. They are assumed to be duplicate entries, since the data has no row-level ID to confirm this.
- **Dates:** `InvoiceDate` was stored as text and was converted to datetime.
- **Missing values:** `CustomerID` is missing in 135,037 rows (~25%) and `Description` in 1,454 rows. These rows are kept; customer-level analysis can only use rows with a customer ID.
- **After cleaning:** 536,641 rows, 8 columns.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Online Retail.csv")

In [ ]:
df.duplicated().sum()   # count fully identical rows (5,268)


np.int64(5268)

In [ ]:
df = df.drop_duplicates()   # assumed to be duplicate entries; only ~1% of rows, so impact is limited


In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])   # parse text to datetime for time-based analysis
df["InvoiceDate"].dtype   # check: datetime64[ns]


dtype('<M8[ns]')

In [ ]:
df.shape   # 536,641 rows after de-duplication

(536641, 8)

In [7]:
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
df.info()   # check column types and non-null counts


<class 'pandas.core.frame.DataFrame'>
Index: 536641 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    536641 non-null  object        
 1   StockCode    536641 non-null  object        
 2   Description  535187 non-null  object        
 3   Quantity     536641 non-null  int64         
 4   InvoiceDate  536641 non-null  datetime64[ns]
 5   UnitPrice    536641 non-null  float64       
 6   CustomerID   401604 non-null  float64       
 7   Country      536641 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 36.8+ MB


In [ ]:
missing_values_count = df.isnull().sum()   # missing values per column
print(missing_values_count)   # Description: 1,454; CustomerID: 135,037

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64


## 2. Rows that should not count as sales

Several groups of rows are not genuine sales. Each group was checked before deciding how to treat it.

| Group | Rows | What it likely is | Treatment |
|---|---|---|---|
| Cancellations (invoice starts with "C") | 9,251 | Customer cancellations or returns | Excluded from gross sales, kept for net sales |
| Negative quantity, not a cancellation | 1,336 | Likely stock write-offs or adjustments: price 0, no customer ID, descriptions such as "damaged" or "thrown away" | Excluded |
| Negative price ("Adjust bad debt") | 2 | Accounting entries | Excluded |
| Price 0, positive quantity | 1,174 | Possibly free gifts, samples, replacements or stock corrections; 1,134 have no customer ID | Excluded from sales (contribute 0 revenue anyway) |
| Manual entries (StockCode "M") | 566 | Manually keyed lines carrying real amounts | Kept in sales; excluded from product rankings |

These groups were identified from the data; the data does not state the reasons, so the explanations above are assumptions.

In [ ]:
df.loc[df.UnitPrice <= 0]   # rows with price <= 0 (2,512 rows)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom
538504,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom
538505,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom
538554,581408,85175,NaN,20,2011-12-08 14:06:00,0.0,NaN,United Kingdom


In [ ]:
df.loc[df.Quantity <= 0]   # rows with quantity <= 0 (10,587 rows)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
541541,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


In [13]:
print(df["InvoiceNo"].dtype)

object


In [ ]:
is_cancel = df["InvoiceNo"].astype(str).str.startswith("C")   # invoice numbers starting with "C" = cancellations
is_cancel.sum()   # 9,251 cancellation rows

np.int64(9251)

In [ ]:
(df["Quantity"] < 0).sum()   # 10,587 rows with negative quantity (no zero-quantity rows exist)


np.int64(10587)

In [ ]:
df.loc[(df["Quantity"] < 0) & (~is_cancel)].head()   # negative quantity but not a cancellation


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom


In [ ]:
odd = df.loc[(df["Quantity"] < 0) & (~is_cancel)]
odd.shape   # 1,336 rows


(1336, 8)

In [ ]:
(odd["Description"].isna()).sum()   # 862 without description


np.int64(862)

In [ ]:
(odd["CustomerID"].isna()).sum()   # all 1,336 without customer ID


np.int64(1336)

In [ ]:
(odd["UnitPrice"] == 0).sum()   # all 1,336 with price 0


np.int64(1336)

In [ ]:
odd["Description"].value_counts().head(10)   # descriptions such as "damaged", "thrown away" suggest stock write-offs


Description
check                     120
damages                    45
damaged                    42
?                          41
sold as set on dotcom      20
Damaged                    14
thrown away                 9
Unsaleable, destroyed.      9
??                          7
damages?                    5
Name: count, dtype: int64

In [ ]:
df.loc[df.UnitPrice < 0]   # only 2 rows; "Adjust bad debt" accounting entries, not sales


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [ ]:
zero_price_have_quantity = df.loc[(df.UnitPrice == 0) & (df.Quantity > 0)]   # price 0 but positive quantity (1,174 rows)
zero_price_have_quantity

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535334,581211,22142,check,14,2011-12-07 18:36:00,0.0,NaN,United Kingdom
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom
538504,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom
538505,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom


In [ ]:
zero_price_have_quantity["CustomerID"].isna().sum()   # 1,134 have no customer ID


np.int64(1134)

In [ ]:
zero_price_customer = zero_price_have_quantity.loc[~zero_price_have_quantity["CustomerID"].isna()]   # 40 rows with a customer ID
zero_price_customer # possibly free gifts, samples or replacements

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647.0,Germany
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16 14:36:00,0.0,16560.0,United Kingdom
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,2010-12-21 13:45:00,0.0,14911.0,EIRE
47068,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
47070,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,2011-01-13 15:10:00,0.0,15107.0,United Kingdom
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,0.0,17560.0,United Kingdom
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,2011-03-23 10:25:00,0.0,13239.0,United Kingdom
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,2011-03-30 12:45:00,0.0,13113.0,United Kingdom
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,2011-04-04 14:42:00,0.0,14410.0,United Kingdom


In [ ]:
m_order = df.loc[df["StockCode"] == "M"]   
m_order# 566 manually entered lines; keep, they carry real amounts

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2239,536569,M,Manual,1,2010-12-01 15:35:00,1.25,16274.0,United Kingdom
2250,536569,M,Manual,1,2010-12-01 15:35:00,18.95,16274.0,United Kingdom
5684,536865,M,Manual,1,2010-12-03 11:28:00,2.55,NaN,United Kingdom
6798,536981,M,Manual,2,2010-12-03 14:26:00,0.85,14723.0,United Kingdom
7976,537077,M,Manual,12,2010-12-05 11:59:00,0.42,17062.0,United Kingdom
...,...,...,...,...,...,...,...,...
531432,580956,M,Manual,4,2011-12-06 14:23:00,1.25,17841.0,United Kingdom
533082,C581009,M,Manual,-1,2011-12-07 09:15:00,125.00,16971.0,United Kingdom
534308,C581145,M,Manual,-1,2011-12-07 13:48:00,9.95,17490.0,United Kingdom
538321,581405,M,Manual,3,2011-12-08 13:50:00,0.42,13521.0,United Kingdom


## 3. Definition of sales

- **Gross sales:** quantity > 0 and price > 0 (524,878 rows). Answers: how much was sold.
- **Net sales:** price > 0, cancellations kept (534,129 rows). Answers: how much revenue remains after cancellations.

Row counts reconcile: 536,641 − 2,512 (price ≤ 0) = 534,129; 534,129 − 9,251 (cancellations) = 524,878.

In [ ]:
gross_sales = df.loc[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
gross_sales.shape   # rows used for gross sales: 524,878

(524878, 8)

In [ ]:
net_sales = df.loc[df["UnitPrice"] > 0]
net_sales.shape   # rows used for net sales (keeps cancellations): 534,129

(534129, 8)

In [ ]:
1 - (len(gross_sales) / len(net_sales))   # cancellations are only ~1.7% of rows

0.017319786044195262

## 4. Cancellations

Cancellations are only ~1.7% of rows but ~8.4% of gross sales value, so they are much larger than average.
The 10 largest cancellations make up ~43% of all cancelled value. They are mostly not ordinary returns:
- Two very large orders (80,995 and 74,215 units) that were probably placed and then cancelled
- Six "AMAZON FEE" lines, likely platform fee entries (assumption)
- One manual adjustment

Excluding these 10 rows, the cancellation rate drops from ~8.4% to ~4.8%.
Limitation: the original sales of the large cancelled orders are still in gross sales, so the adjusted rate is approximate. Other non-product lines (for example "Discount") were not investigated.

In [ ]:
gross = (gross_sales['Quantity'] * gross_sales['UnitPrice']).sum()
net = (net_sales['Quantity'] * net_sales['UnitPrice']).sum()
refund_rate = (gross - net) / gross   
print(refund_rate)# ~8.4% of gross sales value was cancelled

0.08400398628287017


In [ ]:
# Largest cancellations: rank cancellation lines by value to check for unusually large ones
cancel = net_sales.loc[net_sales["InvoiceNo"].astype(str).str.startswith("C")].copy()   # keep cancellation lines only
cancel["amount"] = cancel["Quantity"] * cancel["UnitPrice"]   # cancelled value per line (negative)

top10 = cancel.sort_values("amount").head(10)   # 10 largest cancellations (most negative amounts first)
top10[["InvoiceNo", "Description", "Quantity", "UnitPrice", "CustomerID", "amount"]]

,InvoiceNo,Description,Quantity,UnitPrice,CustomerID,amount
540422,C581484,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,16446.0,-168469.60
61624,C541433,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346.0,-77183.60
222681,C556445,Manual,-1,38970.00,15098.0,-38970.00
524602,C580605,AMAZON FEE,-1,17836.46,NaN,-17836.46
43702,C540117,AMAZON FEE,-1,16888.02,NaN,-16888.02
43703,C540118,AMAZON FEE,-1,16453.71,NaN,-16453.71
15016,C537630,AMAZON FEE,-1,13541.33,NaN,-13541.33
16356,C537651,AMAZON FEE,-1,13541.33,NaN,-13541.33
16232,C537644,AMAZON FEE,-1,13474.79,NaN,-13474.79
524601,C580604,AMAZON FEE,-1,11586.50,NaN,-11586.50


In [ ]:
# How much do the 10 largest cancellations distort the cancellation rate?
cancel_total = cancel["amount"].sum()   # total cancelled value (negative)
top10_total = top10["amount"].sum()     # cancelled value of the top 10 lines (negative)

print(top10_total / cancel_total)   # share of the top 10 in all cancelled value (~43%)

adj_rate = -(cancel_total - top10_total) / gross   # cancellation rate without the top 10 (as a share of gross sales)
print(adj_rate)   # ~4.8%, compared with ~8.4% for all cancellations

0.4339531725176811
0.04755018993128688


In [33]:
missing = df["CustomerID"].isna().sum()
print(missing, missing / len(df))

135037 0.2516337737891812


## 5. Sales by country and product

**Country.** The UK accounts for ~84.6% of gross sales. The next largest markets (Netherlands, EIRE, Germany, France) are each only a few percent. The business depends heavily on the UK. This ranking uses totals only and says nothing about growth.

**Products.** Postage and "Manual" appeared in the revenue top 10 and were excluded because they are not products.
By units, the top two products (about 159,000 units combined) were the large orders that were cancelled. Using net units, they drop out of the top 10. Rankings by units should therefore use net figures, otherwise "ordered and cancelled" items look like best sellers.

**Revenue ranking, gross vs net.** The same two products also sit in the gross revenue top 10 (ranks 2 and 6, after excluding postage and "Manual"). Once cancellations are deducted they drop out, and two other products enter ranks 9 and 10. The best-selling product by net revenue is the Regency Cakestand 3 Tier (about £164k) in both views. Product rankings should therefore be based on net figures, otherwise large orders that were later cancelled appear as top products.

In [ ]:
gross_sales = gross_sales.copy()
gross_sales["amount"] = gross_sales["Quantity"] * gross_sales["UnitPrice"]   # line revenue


In [ ]:
gross_rank_country = gross_sales.groupby("Country").amount.sum().sort_values(ascending=False).head(10)
gross_rank_country

Country
United Kingdom    9001744.094
Netherlands        285446.340
EIRE               283140.520
Germany            228678.400
France             209625.370
Australia          138453.810
Spain               61558.560
Switzerland         57067.600
Belgium             41196.340
Sweden              38367.830
Name: amount, dtype: float64

In [ ]:
gross_rank_country.loc["United Kingdom"] / gross   # UK share of gross sales: ~84.6%


np.float64(0.8458607751590557)

In [ ]:
gross_rank_qty = gross_sales.groupby("Description").Quantity.sum().sort_values(ascending=False).head(10)   # top 10 by units sold
gross_rank_amount = gross_sales.groupby('Description').amount.sum().sort_values(ascending=False).head(10)   # top 10 by amount

In [41]:
gross_rank_qty

Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        78033
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54951
JUMBO BAG RED RETROSPOT               48371
WHITE HANGING HEART T-LIGHT HOLDER    37872
POPCORN HOLDER                        36749
PACK OF 72 RETROSPOT CAKE CASES       36396
ASSORTED COLOUR BIRD ORNAMENT         36362
RABBIT NIGHT LIGHT                    30739
MINI PAINT SET VINTAGE                26633
Name: Quantity, dtype: int64

In [42]:
gross_rank_amount

Description
DOTCOM POSTAGE                        206248.77
REGENCY CAKESTAND 3 TIER              174156.54
PAPER CRAFT , LITTLE BIRDIE           168469.60
WHITE HANGING HEART T-LIGHT HOLDER    106236.72
PARTY BUNTING                          99445.23
JUMBO BAG RED RETROSPOT                94159.81
MEDIUM CERAMIC TOP STORAGE JAR         81700.92
POSTAGE                                78101.88
Manual                                 77752.82
RABBIT NIGHT LIGHT                     66870.03
Name: amount, dtype: float64

In [ ]:
non_product = ["DOTCOM POSTAGE", "POSTAGE", "Manual"]   # shipping charges and manual entries are not products

is_non_product = gross_sales["Description"].isin(non_product)
products_only = gross_sales.loc[~is_non_product]   # "~" flips the mask: keep rows NOT in the list

gross_rank_amount = products_only.groupby("Description").amount.sum().sort_values(ascending=False).head(10)
gross_rank_amount

Description
REGENCY CAKESTAND 3 TIER              174156.54
PAPER CRAFT , LITTLE BIRDIE           168469.60
WHITE HANGING HEART T-LIGHT HOLDER    106236.72
PARTY BUNTING                          99445.23
JUMBO BAG RED RETROSPOT                94159.81
MEDIUM CERAMIC TOP STORAGE JAR         81700.92
RABBIT NIGHT LIGHT                     66870.03
PAPER CHAIN KIT 50'S CHRISTMAS         64875.59
ASSORTED COLOUR BIRD ORNAMENT          58927.62
CHILLI LIGHTS                          54096.36
Name: amount, dtype: float64

In [46]:
net_sales = net_sales.copy()
net_sales["amount"] = net_sales["Quantity"] * net_sales["UnitPrice"]
net_rank_amount = (net_sales.loc[~net_sales["Description"].isin(non_product)]
                   .groupby("Description").amount.sum()
                   .sort_values(ascending=False).head(10))
net_rank_amount

Description
REGENCY CAKESTAND 3 TIER              164459.49
WHITE HANGING HEART T-LIGHT HOLDER     99612.42
PARTY BUNTING                          98243.88
JUMBO BAG RED RETROSPOT                92175.79
RABBIT NIGHT LIGHT                     66661.63
PAPER CHAIN KIT 50'S CHRISTMAS         63715.24
ASSORTED COLOUR BIRD ORNAMENT          58792.42
CHILLI LIGHTS                          53746.66
SPOTTY BUNTING                         42030.67
JUMBO BAG PINK POLKADOT                41584.43
Name: amount, dtype: float64

In [ ]:
net_rank_qty = net_sales.groupby("Description").Quantity.sum().sort_values(ascending=False).head(10)   # top 10 by net units (returns deducted)
net_rank_qty

Description
WORLD WAR 2 GLIDERS ASSTD DESIGNS     53751
JUMBO BAG RED RETROSPOT               47256
POPCORN HOLDER                        36322
ASSORTED COLOUR BIRD ORNAMENT         36282
PACK OF 72 RETROSPOT CAKE CASES       36016
WHITE HANGING HEART T-LIGHT HOLDER    35294
RABBIT NIGHT LIGHT                    30631
MINI PAINT SET VINTAGE                26437
PACK OF 12 LONDON TISSUES             26095
PACK OF 60 PINK PAISLEY CAKE CASES    24719
Name: Quantity, dtype: int64